In [8]:
"""
Script de prueba rápida: Configuración de velocidad dedo a dedo.
Hereda de la librería base para usar su conexión nativa y forzar la escritura Modbus.
"""

import time
from inspire_rh56dftp.client import InspireHand

# Dirección Modbus para configurar la velocidad según el datasheet
SPEED_SET = 1522

class InspireHandVelocidad(InspireHand):
    def send_speed(self, speed_values):
        """Envía 6 valores de velocidad a la mano (rango 0 a 1000)."""
        if self.dead:
            return False
            
        # Aseguramos que los valores estén en el rango permitido (0 a 1000 para todos)
        clamped = [max(0, min(1000, int(v))) for v in speed_values]
        
        try:
            # Inyectamos la trama directamente al registro 1522
            self._write_and_ack(SPEED_SET, clamped)
            return True
        except Exception as e:
            print(f"Error enviando velocidades: {e}")
            return False

def main():
    IP_MANO = "192.168.123.211"
    print(f"Conectando a la mano en {IP_MANO}...")
    
    mano = InspireHandVelocidad(ip=IP_MANO)
    mano._write_and_ack(1498, [1000, 1000, 1000, 1000, 1500, 1000])
    # Llevamos la mano a la posición abierta (1000) primero con velocidad normal
    print("Abriendo la mano por completo...")
    mano.open_hand()
    time.sleep(2)

    # 1. Definir velocidades: [Meñique, Anular, Corazón, Índice, Flexión pulgar, Rot. pulgar]
    # Usamos velocidades escalonadas para ver cómo cada dedo llega a destiempo.
    # Meñique muy lento (100), Índice normal (800)
    velocidades_prueba = [100, 300, 500, 800, 1000, 1000]

    # 2. Definir una postura de cierre completo (0) para tener el mayor recorrido visible
    angulos_prueba = [0, 0, 0, 0, 0, 1000]
    time.sleep(2)
    print(f"\nEscribiendo registro {SPEED_SET} con velocidades: {velocidades_prueba}...")
    exito_velocidad = mano.send_speed(velocidades_prueba)
    
    if exito_velocidad:
        time.sleep(0.1) # Pausa necesaria para que el controlador asimile el registro
        
        print(f"Cerrando dedos a la posición: {angulos_prueba}")
        mano.send_raw(angulos_prueba)
        
        print("\n>> Observa la mano... los dedos deberían flexionarse a distintas velocidades.")
        time.sleep(6) # Damos tiempo suficiente porque los dedos más lentos tardarán en llegar
        
        # Restaurar la velocidad máxima por seguridad antes de salir
        print("Restaurando velocidad normal (1000) y abriendo la mano para finalizar...")
        mano.send_speed([1000, 1000, 1000, 1000, 1000, 1000])
        time.sleep(0.1)
        mano.open_hand()
        time.sleep(1.5)
    else:
        print("No se pudo configurar la velocidad por un error de comunicación.")

    mano.close()
    print("Prueba finalizada. Conexión cerrada.")

if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211...
Abriendo la mano por completo...

Escribiendo registro 1522 con velocidades: [100, 300, 500, 800, 1000, 1000]...
Cerrando dedos a la posición: [0, 0, 0, 0, 0, 1000]

>> Observa la mano... los dedos deberían flexionarse a distintas velocidades.
Restaurando velocidad normal (1000) y abriendo la mano para finalizar...
Prueba finalizada. Conexión cerrada.
